# The Machine-Learning Workflow, End to End

This notebook is the whole course in miniature. We take one small data set and
walk through every stage of a machine-learning problem once, so that the shape
of the process is familiar before any single stage is studied in depth.

**The question.** A company advertises a product on TV, on radio, and in
newspapers, across 200 different markets. Given the advertising budget in a
market we have never seen, what sales should we predict?

**The five stages.** Every method in this course is an answer to one of these
questions.

| stage | the question it answers | our answer today |
|---|---|---|
| Data | What do we have, and what is one row? | 200 markets: three budgets and one sales figure |
| Model | Which functions are we willing to consider? | Those linear in the three budgets |
| Loss | What does it cost to be wrong? | Squared error |
| Optimization | How do we find the best function we allowed? | Solve the normal equations |
| Evaluation | How well will it do on markets we have not seen? | Error on a held-out 30% |

**By the end you should be able to** load a data set, split it honestly, fit a
linear model by writing the linear algebra yourself, measure error on held-out
data, and explain why a more flexible model can score better on the data it was
fitted to and worse on everything else.

Each stage gets a lecture of its own later in the term. This notebook is the
map, so nothing below assumes material we have not covered.

---

## 1. The data

`Advertising` is the running example from ISLP. One row is **one market** — a
geographic sales region. Four numbers describe it:

- `TV`, `radio`, `newspaper` — the advertising budget in that market, in
  thousands of dollars, for each of the three channels;
- `sales` — units sold in that market, in thousands.

The first three are the **features**, written $x$: the measurements available
at prediction time. The last is the **response**, written $y$: the quantity we
want to predict.

`Advertising.csv` sits in this folder, beside the notebook, so the cell below
just opens it.

`SEED` fixes the random shuffle used later, so your numbers will match the ones
shown in class.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 26501
rng = np.random.default_rng(SEED)
advertising = pd.read_csv('Advertising.csv', index_col=0)
advertising.head()

`head()` prints the first five rows. Check that the index is the market number
and that the four columns are the ones described above.

The next cell summarizes all 200 rows, and then asserts two facts about the
file.

In [ ]:
print(advertising.describe().round(2))
assert len(advertising) == 200
assert {'TV', 'radio', 'newspaper', 'sales'} <= set(advertising.columns)

Two things in that summary matter before any modeling starts.

- The three budgets live on **very different scales**. TV budgets run into the
  hundreds while newspaper budgets are much smaller. Raw coefficients on these
  columns would not be comparable, which is why we standardize in the next
  section.
- `sales` stays well away from zero, so the model needs an **intercept**: a
  market with no advertising at all still sells something.

The two `assert` lines are a habit worth copying. They state what the rest of
the notebook is relying on, and they fail loudly and immediately if the file
ever changes underneath you. A silent wrong answer is much more expensive than
a crash.

---

## 2. A train/test split, and least squares written out

This is the heart of the notebook: four decisions, made in order.

**The split.** We shuffle the 200 markets and keep 70% for fitting and 30% for
evaluation. The held-out 30% stands in for markets we have never seen. Without
a split we could only report how well the model does on data it has already
been shown, which tells us almost nothing about the next market.

**Standardization.** Each feature is centered and rescaled to have mean zero
and standard deviation one. The mean and standard deviation are computed from
the **training rows only**, and those same two numbers are then applied to the
test rows. Computing them from all 200 rows would let the test markets
influence the fit, which is data leakage: the held-out score would be
measured on rows that had already helped choose the scaling.

**The model.** A column of ones is placed in front of the features, so that the
matrix product $X\beta$ carries an intercept. The model class is every function
of the form

$$f(x) \;=\; \beta_0 + \beta_1 x_{\text{TV}} + \beta_2 x_{\text{radio}} + \beta_3 x_{\text{news}}.$$

Choosing a model means choosing which functions we are willing to consider.
Here we allow exactly these, and nothing curved.

**The loss and the optimizer.** Squared error, averaged over rows, gives the
mean squared error. Because that loss is convex and quadratic in $\beta$, the
minimizer is the solution of the **normal equations**

$$X^\top X \beta \;=\; X^\top y,$$

which `np.linalg.solve` handles in one line. Later lectures replace this closed
form with gradient descent, which is what you need once the model is
complicated enough that no closed form exists.

In [ ]:
order = rng.permutation(len(advertising))
cut = int(.70 * len(order))
train, test = order[:cut], order[cut:]
features = ['TV', 'radio', 'newspaper']
X_raw = advertising[features].to_numpy(float)
y = advertising['sales'].to_numpy(float)

mu, sd = X_raw[train].mean(0), X_raw[train].std(0)
X_train = np.column_stack([np.ones(len(train)), (X_raw[train] - mu) / sd])
X_test = np.column_stack([np.ones(len(test)), (X_raw[test] - mu) / sd])

def least_squares(X, y):
    return np.linalg.solve(X.T @ X, X.T @ y)

def mse(y, prediction):
    return np.mean((y - prediction) ** 2)

beta = least_squares(X_train, y[train])
print('coefficients [intercept, TV, radio, newspaper]:', beta.round(3))
print('training MSE:', mse(y[train], X_train @ beta).round(3))
print('test MSE:', mse(y[test], X_test @ beta).round(3))

Read the three lines of output as follows.

- The **coefficients** are on the standardized scale, so they can be compared
  directly with one another: each says how much predicted sales move when that
  budget rises by one standard deviation, holding the others fixed. TV and
  radio carry the signal; the newspaper coefficient sits near zero.
- **Training MSE** measures the fit on the rows that were used to choose
  $\beta$. It is optimistic by construction.
- **Test MSE** measures the fit on rows the fitting procedure never saw. This
  is the number that estimates future performance, and it is the one to report.

A caution about the coefficients: they describe associations in this data set.
They do not establish that moving a budget would cause sales to change. That
distinction returns in Lecture 3.

---

## 3. Flexibility and held-out error

The model above was a straight line in each feature. What happens when we let
it bend?

This section uses a single feature, `TV`, and fits polynomials of degree 1
through 15. The degree is a dial for **flexibility**: degree 1 is a straight
line, and degree 15 is a curve that can wind its way through almost any set of
points.

Predict the shape of the two curves before running the cell. Training error can
only fall as the degree rises, because every degree-$d$ polynomial can do
anything a degree-$(d-1)$ polynomial can do. Test error is under no such
obligation.

**Reading the code.** Three things in the cell are worth pausing on.

`polynomial_design` builds a design matrix whose columns are the powers
$1, z, z^2, \dots, z^d$. Fitting a polynomial is therefore still *linear*
regression — linear in the coefficients, even though the resulting curve bends.
That is the trick that lets one estimator cover a whole family of shapes, and it
comes back in Lecture 6 under the name *basis expansion*.

The `center` and `scale` inside it are a numerical necessity rather than a
modeling choice. Raw TV budgets run up to 296, and $296^{15}$ is about
$10^{37}$ — large enough to swamp the smaller columns and wreck the arithmetic.
Centering and rescaling first keeps every power in a comfortable range. Note
that the two constants come from the training rows, for the same reason as in
Section 2.

`np.linalg.lstsq` replaces the `np.linalg.solve` we used earlier. At degree 12
or 15 the columns of $\Phi$ are nearly linearly dependent — successive powers of
the same numbers look increasingly alike — and the normal equations become too
ill-conditioned to solve directly. `lstsq` works from the SVD instead, which
degrades gracefully where `solve` would fail outright. Lecture 4 explains what
"nearly dependent columns" does to a fit and what to do about it.

In [ ]:
x = advertising['TV'].to_numpy(float)

def polynomial_design(values, degree, center, scale):
    z = (values - center) / scale
    return np.column_stack([z ** j for j in range(degree + 1)])

center, scale = x[train].mean(), x[train].std()
degrees = np.arange(1, 16)
training_error, test_error = [], []
for degree in degrees:
    Phi_train = polynomial_design(x[train], degree, center, scale)
    Phi_test = polynomial_design(x[test], degree, center, scale)
    coefficients = np.linalg.lstsq(Phi_train, y[train], rcond=None)[0]
    training_error.append(mse(y[train], Phi_train @ coefficients))
    test_error.append(mse(y[test], Phi_test @ coefficients))

plt.plot(degrees, training_error, 'o-', label='training')
plt.plot(degrees, test_error, 'o-', label='test')
plt.xlabel('polynomial degree'); plt.ylabel('MSE'); plt.legend();
print('best held-out degree:', degrees[np.argmin(test_error)])

The gap that opens between the two curves is **overfitting**, made visible. A
flexible model starts fitting the noise particular to the training markets, and
noise is by definition unrepeatable, so the extra flexibility that lowers
training error raises error everywhere else.

The degree printed under the plot is the one this procedure would select. Notice
that we chose it by looking at the test set, which is itself a mild form of
leakage: the reported test error for the winning degree is now slightly
optimistic. The repair is a third split, called a validation set, and it arrives
in Lecture 4.

---

## 4. The same fit, from a library

Everything so far was written out in NumPy, one line of linear algebra at a
time. `scikit-learn` performs the same computation behind a single method call.

The cell below fits `LinearRegression` on the same standardized design matrix
and checks the two against each other. `fit_intercept=False` is there because
our matrix already carries its own column of ones.

In [ ]:
from sklearn.linear_model import LinearRegression

library = LinearRegression(fit_intercept=False).fit(X_train, y[train])
library_prediction = library.predict(X_test)
print('scikit-learn test MSE:', mse(y[test], library_prediction).round(3))
print('maximum prediction difference:', np.max(np.abs(library_prediction - X_test @ beta)))
assert np.allclose(library_prediction, X_test @ beta)

The largest disagreement is around $10^{-14}$, which is floating-point rounding
rather than a real difference. The two calculations are the same calculation,
and the assertion confirms it.

That is the point of running both. `LinearRegression` is doing the arithmetic
you just wrote out, and being able to reproduce a library's answer by hand is
what will let you tell a bug from a result once the numbers get strange enough
to doubt.

---

## What to take away

- A machine-learning problem is **five decisions**: data, model, loss,
  optimizer, and evaluation. Every lecture this term develops one of them.
- The number worth reporting is the error on data the fitting procedure never
  saw.
- Raising flexibility always improves training error, and past some point it
  damages test error. The gap between the two curves is the whole subject of
  Lecture 6.
- Anything estimated from data — a mean, a standard deviation, a polynomial
  degree — must be estimated from the training rows alone.

**Next.** Lecture 2 develops the mathematical objects this notebook used
without naming them: vectors and norms, the design matrix and its geometry,
gradients, and the probability needed to say what "noise" means.